# PWM Quickstart: Physics World Model in 5 Minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/integritynoble/Physics_World_Model/blob/master/examples/PWM_Quickstart.ipynb)

This notebook demonstrates the core PWM workflow:
1. **Simulate** measurements from a physics forward model
2. **Reconstruct** the original signal using a solver
3. **Diagnose** reconstruction quality

No local setup needed -- just click the Colab badge above and run all cells.

## 1. Install PWM

In [ ]:
# Install PWM core (takes ~30 seconds)
!pip install -q git+https://github.com/integritynoble/Physics_World_Model.git#subdirectory=packages/pwm_core

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('PWM installed successfully!')

## 2. Hello World: Reconstruct a Blurred Image

The simplest computational imaging problem: deblur an image when you know the blur kernel (PSF).

**Forward model:** `y = H * x + noise` (convolution + noise)

**Goal:** Recover `x` from `y` given `H`.

In [ ]:
# Create a simple test image (checkerboard pattern)
def make_checkerboard(size=64, block=8):
    """Generate a checkerboard test image."""
    x = np.zeros((size, size), dtype=np.float64)
    for i in range(size):
        for j in range(size):
            if (i // block + j // block) % 2 == 0:
                x[i, j] = 1.0
    return x

x_true = make_checkerboard(64, 8)
print(f'Ground truth shape: {x_true.shape}, range: [{x_true.min():.1f}, {x_true.max():.1f}]')

In [ ]:
# Define a simple physics operator (Gaussian blur)
from scipy.ndimage import gaussian_filter

class BlurOperator:
    """Simple Gaussian blur forward model."""
    def __init__(self, sigma=2.0):
        self.sigma = sigma
        self.x_shape = (64, 64)
        self.y_shape = (64, 64)

    def forward(self, x):
        """Apply blur: y = H(x)."""
        return gaussian_filter(x, self.sigma)

    def adjoint(self, y):
        """Apply adjoint (blur is self-adjoint for symmetric PSF)."""
        return gaussian_filter(y, self.sigma)

# Simulate measurement
physics = BlurOperator(sigma=2.0)
noise_level = 0.01
rng = np.random.default_rng(42)
y = physics.forward(x_true) + rng.normal(0, noise_level, x_true.shape)

print(f'Measurement y: shape={y.shape}, SNR~{10*np.log10(np.var(x_true)/noise_level**2):.1f} dB')

In [ ]:
# Reconstruct using gradient descent (the core PWM solver protocol)
def reconstruct_gradient_descent(y, physics, iters=200, step_size=0.5):
    """Simple gradient descent: min_x ||H(x) - y||^2."""
    x_hat = physics.adjoint(y)  # Initialize with A^T y

    for i in range(iters):
        residual = physics.forward(x_hat) - y
        gradient = physics.adjoint(residual)
        x_hat = x_hat - step_size * gradient

    return x_hat

x_hat = reconstruct_gradient_descent(y, physics, iters=200, step_size=0.5)

# Compute metrics
mse = np.mean((x_hat - x_true) ** 2)
psnr = 10 * np.log10(1.0 / mse) if mse > 0 else float('inf')
print(f'Reconstruction PSNR: {psnr:.2f} dB')

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(x_true, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(y, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Blurred Measurement')
axes[1].axis('off')

axes[2].imshow(np.clip(x_hat, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Reconstructed (PSNR={psnr:.1f} dB)')
axes[2].axis('off')

plt.suptitle('PWM: Simulate -> Measure -> Reconstruct', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Write Your Own Solver (Level 1 Contribution)

Any function that takes `(y, physics, cfg)` and returns `(x_hat, info)` is a valid PWM solver.

Here's the **exact protocol** -- copy this template, replace the algorithm, and you have a contribution.

In [ ]:
def run_my_solver(y, physics, cfg):
    """Your solver here! Replace the algorithm below.

    Parameters
    ----------
    y : np.ndarray       -- measurements
    physics : operator   -- has .forward(), .adjoint(), .x_shape, .y_shape
    cfg : dict           -- solver config (iters, step_size, etc.)

    Returns
    -------
    x_hat : np.ndarray   -- reconstruction
    info : dict          -- metadata
    """
    iters = cfg.get('iters', 100)
    step_size = cfg.get('step_size', 0.01)

    # --- YOUR ALGORITHM HERE ---
    x_hat = physics.adjoint(y)
    for i in range(iters):
        residual = physics.forward(x_hat) - y
        gradient = physics.adjoint(residual)
        x_hat = x_hat - step_size * gradient
    # --- END YOUR ALGORITHM ---

    return x_hat, {'solver': 'my_solver', 'iters': iters}

# Test it
x_hat2, info = run_my_solver(y, physics, {'iters': 200, 'step_size': 0.5})
mse2 = np.mean((x_hat2 - x_true) ** 2)
psnr2 = 10 * np.log10(1.0 / mse2) if mse2 > 0 else float('inf')
print(f'Your solver: PSNR = {psnr2:.2f} dB')
print(f'Info: {info}')

## 4. The 4-Scenario Protocol (How PWM Evaluates Methods)

PWM tests every solver under 4 scenarios to measure the impact of operator mismatch:

| Scenario | Measurement | Reconstruction | Purpose |
|----------|-------------|----------------|---------|
| I (Ideal) | True H | True H | Oracle upper bound |
| II (Mismatch) | True H | Wrong H | How bad is mismatch? |
| III (Corrected) | True H | Calibrated H | Does calibration help? |

**Key metric:** Recovery ratio rho = (PSNR_III - PSNR_II) / (PSNR_I - PSNR_II)

In [ ]:
# Demonstrate the 4-scenario protocol

# True physics: sigma=2.0
H_true = BlurOperator(sigma=2.0)

# Mismatched physics: wrong sigma (the real-world problem)
H_wrong = BlurOperator(sigma=3.0)

# Calibrated physics: partially corrected
H_calib = BlurOperator(sigma=2.2)  # Close but not perfect

# Generate measurement with true physics
y = H_true.forward(x_true) + rng.normal(0, 0.01, x_true.shape)

# Scenario I: Ideal (true H for both)
x_I = reconstruct_gradient_descent(y, H_true, iters=200, step_size=0.5)
psnr_I = 10 * np.log10(1.0 / np.mean((x_I - x_true)**2))

# Scenario II: Mismatch (wrong H for reconstruction)
x_II = reconstruct_gradient_descent(y, H_wrong, iters=200, step_size=0.5)
psnr_II = 10 * np.log10(1.0 / np.mean((x_II - x_true)**2))

# Scenario III: Corrected (calibrated H)
x_III = reconstruct_gradient_descent(y, H_calib, iters=200, step_size=0.5)
psnr_III = 10 * np.log10(1.0 / np.mean((x_III - x_true)**2))

# Recovery ratio
rho = (psnr_III - psnr_II) / (psnr_I - psnr_II) if (psnr_I - psnr_II) > 0 else 0

print(f'Scenario I  (Ideal):     PSNR = {psnr_I:.2f} dB')
print(f'Scenario II (Mismatch):  PSNR = {psnr_II:.2f} dB')
print(f'Scenario III (Corrected): PSNR = {psnr_III:.2f} dB')
print(f'\nRecovery ratio rho = {rho:.2f}')
print(f'  (rho=1.0 means calibration fully closes the gap)')

In [ ]:
# Visualize all 3 scenarios
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(x_true, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(np.clip(x_I, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'I: Ideal ({psnr_I:.1f} dB)')
axes[1].axis('off')

axes[2].imshow(np.clip(x_II, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'II: Mismatch ({psnr_II:.1f} dB)')
axes[2].axis('off')

axes[3].imshow(np.clip(x_III, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[3].set_title(f'III: Corrected ({psnr_III:.1f} dB)')
axes[3].axis('off')

plt.suptitle(f'4-Scenario Protocol  |  Recovery ratio rho = {rho:.2f}', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Next Steps

### Contribute a Solver (Level 1, ~1 day)
```bash
# On your local machine:
pip install -e packages/pwm_core
pwm scaffold solver my_solver      # generates template
# Edit contrib/solvers/my_solver/solver.py
pwm evaluate --sandbox --modality widefield --solver my_solver
# Submit PR -- auto-merges in 48h if CI passes
```

### Compete on the Leaderboard (No PR needed)
```bash
pwm evaluate --modality cassi --solver my_solver --output ./results
pwm submit ./results/runbundle.zip
```

### Links
- [GitHub Repository](https://github.com/integritynoble/Physics_World_Model)
- [Contributing Guide](https://github.com/integritynoble/Physics_World_Model/blob/master/CONTRIBUTING.md)
- [Weekly Challenges](https://github.com/integritynoble/Physics_World_Model/tree/master/community/challenges)
- [GitHub Discussions (Q&A)](https://github.com/integritynoble/Physics_World_Model/discussions)